In [6]:
#PACKAGES
import time
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains as AC
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
import pandas as pd
import numpy as np

In [37]:
class Stock_Scraper:
    def __init__(self, timeout=10):
        # 1. This method runs first and creates self.driver
        self.init_driver_options()
        
        # 2. FIXED: Changed 'driver' to 'self.driver' so WebDriverWait can hook into it
        self.wait = WebDriverWait(self.driver, timeout=timeout)
        self.data = []

        self.actions = AC(self.driver)

    def init_driver_options(self):
        options = Options()
        
        # 1. Stop Yahoo from locking up the browser thread with background processes
        options.add_argument("--disable-gpu")
        options.add_argument("--disable-dev-shm-usage")  # Prevents memory crashes
        options.add_argument("--no-sandbox")
        
        # 2. Strong anti-detection headers to prevent aggressive bot challenges
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
        
        # 3. Optimization: Force Chrome to return control to Python immediately after HTML arrives
        options.page_load_strategy = 'eager'  # Stops Selenium from waiting for heavy tracker scripts
        
        self.driver = webdriver.Chrome(options=options)
        self.driver.maximize_window()
        
        # Evade navigator detection hooks
        self.driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
            "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
        })

    def wait_for_load(self):
        try:
            # Give a small brief pause for basic elements to settle
            time.sleep(2) 
            self.wait.until(
                lambda d: d.execute_script("return document.readyState") == 'interactive'
            )
            page_title = self.driver.title
            print(f"✅Successfully loaded➡️\"{page_title}\"!\n")
            
        except Exception as e:
            print(f"❌ Automation collapsed. Error detail: {e}\n")

    def access_url(self, url):
        self.driver.get(url)
        self.wait_for_load()

    def access_most_active_stocks(self):
        
        market_menue = self.wait.until(
             EC.presence_of_element_located((By.XPATH, '//*[@id="navigation-container"]/ol/li[2]/a/div'))
        )
        self.actions.move_to_element(market_menue).perform()
        
        stocks = self.wait.until(
            EC.presence_of_element_located((By.XPATH, "//span[normalize-space()='Stocks']"))
        )
        self.actions.move_to_element(stocks).perform()
    
        most_active = self.wait.until(
            EC.element_to_be_clickable((By.XPATH, "//a[@href='https://finance.yahoo.com/markets/stocks/most-active/']//span[@class='item-content yf-ivjzi2'][normalize-space()='Most Active']"))
        )
        self.actions.click(most_active).perform()
    
        self.wait_for_load()

    def close(self):
        """Safely closes the browser window to prevent hanging memory tracks."""
        if hasattr(self, 'driver') and self.driver:
            self.driver.quit()


    def extract_stocks(self):
        while True:
            #Scraping Data
            try:
                self.wait.until(
                    EC.presence_of_element_located((By.TAG_NAME,'table'))
                )
    
                rows=self.driver.find_elements(By.CSS_SELECTOR,'table tbody tr')
    
                for row in rows:
                    values=row.find_elements(By.TAG_NAME,'td')
                    stock={
                        'symbol':values[0].text,
                        'name':values[1].text,
                        'price':values[3].text,
                        'change':values[4].text,
                        'change_per':values[5].text,
                        'volume':values[6].text,
                        'AVG_Volume':values[7].text,
                        'market_cap':values[8].text,
                        'PE_ratio_TTM':values[9].text,
                        'WK_change':values[10].text,
                        'WK_Range':values[11].text
                    }
                    self.data.append(stock)
    
                
            except Exception as e:
                print(f'❌Could not scrape the Data due to this➡️;{e}')
            
            
            #clicking Button
            try:
                next_button=self.wait.until(
                    EC.element_to_be_clickable((By.XPATH,'//*[@id="main-content-wrapper"]/section[1]/div/div[4]/div[3]/button[3]'))
                )
    
            except:
                print(f'✅Successfully Scraed the data from the website.')
                break
                
            else:
                self.actions.click(next_button).perform()
                time.sleep(2)
                
    def clean_and_store_data(self,filename='temp'):
            # Here we are cleaning and loading the data into teh excel file.
        
        df=(
            
            pd.DataFrame(self.data)
            .apply(lambda col:col.str.strip() if col.dtype == 'object' else col)
            #.price.str.extract(r'([^0-9.])',expand=False)
            .assign(
                price=lambda x:pd.to_numeric(x['price'].astype(str).replace(',','',regex=True),errors='coerce'),
                change=lambda x:pd.to_numeric(x['change'].str.replace('+','')),
                change_per=lambda x:pd.to_numeric(x['change_per'].str.replace('+','').str.replace('%','')),
                volume=lambda x:pd.to_numeric(x['volume'].str.replace('M','')),
                AVG_Volume=lambda x:pd.to_numeric(x['AVG_Volume'].str.replace(',','').str.replace('M','')),
                market_cap=lambda x:x['market_cap'].apply(lambda val:float(val.replace('B',''))if "B" in val else float(val.replace('T',''))*1000),
                PE_ratio_TTM=lambda x:pd.to_numeric(x['PE_ratio_TTM'],errors='coerce'),
                WK_change=lambda x:pd.to_numeric(x['WK_change'].str.replace('+','').str.replace('%','')),
                Wk_range_start=lambda x:pd.to_numeric(x['WK_Range'].str.split('\n').str.get(0)),
                WK_range_end=lambda x:pd.to_numeric(x['WK_Range'].str.split('\n').str.get(1).str.replace(',',''))

                    )
        .drop(columns='WK_Range')
    
        .rename(columns={
            'price':'price_usd',
            'AVG_Volume':'AVG_Volume_M',
            'market_cap':'market_cap_B'})
        .rename(columns=str.lower)
            )
        df.to_excel(f'{filename}.xlsx',index=False)
        

In [38]:
# Execution block wrapper
if __name__ == "__main__":
    scraper = Stock_Scraper(5)
    url="https://finance.yahoo.com/"
    try:
        scraper.access_url(url)
        scraper.access_most_active_stocks()
        scraper.extract_stocks()
        scraper.clean_and_store_data('Yahoo_active_stocks')
        
    finally:
        scraper.close()

✅Successfully loaded➡️"Yahoo Finance - Stock Market Live, Quotes, Business & Finance News"!

✅Successfully loaded➡️"Most Active Stocks: US stocks with the highest trading volume today - Yahoo Finance"!

✅Successfully Scraed the data from the website.


ModuleNotFoundError: No module named 'openpyxl'

In [39]:
!pip install pyexcel-xlsx

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------- ------------------------------- 1/5 [pyexcel-io]
   -------- ------------------------------- 1/5 [pyexcel-io]
   ------------------------ --------------- 3/5 [openpyxl]
   ------------------------ --------------- 3/5 [openpyxl]
   ------------------------ --------------- 3/5 [openpyxl]
   ------------------------ --------------- 3/5 [openpyxl]
   ------------------------ --------------- 3/5 [openpyxl]
   ------------------------ --------------- 3/5 [openpyxl]
   ------------------------ --------------- 3/5 [openpyxl]
   ------------------------ --------------- 3/5 [openpyxl]
   ------------------------ --------------- 3/5 [openpyxl]
   ------------------------ --------------- 3/5 [openpyxl]
   -----------------------------------